In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.4906696677207947
epoch:  1, loss: 0.29486003518104553
epoch:  2, loss: 0.1852627545595169
epoch:  3, loss: 0.12054410576820374
epoch:  4, loss: 0.08271906524896622
epoch:  5, loss: 0.061466675251722336
epoch:  6, loss: 0.049846868962049484
epoch:  7, loss: 0.043637435883283615
epoch:  8, loss: 0.04038110002875328
epoch:  9, loss: 0.03869851678609848
epoch:  10, loss: 0.037838518619537354
epoch:  11, loss: 0.037402085959911346
epoch:  12, loss: 0.03718128427863121
epoch:  13, loss: 0.03706935793161392
epoch:  14, loss: 0.0370120145380497
epoch:  15, loss: 0.03698192164301872
epoch:  16, loss: 0.03696533665060997
epoch:  17, loss: 0.03695548698306084
epoch:  18, loss: 0.0369434729218483
epoch:  19, loss: 0.03692792356014252
epoch:  20, loss: 0.03691885992884636
epoch:  21, loss: 0.03690869361162186
epoch:  22, loss: 0.036893799901008606
epoch:  23, loss: 0.036885011941194534
epoch:  24, loss: 0.0368742011487484
epoch:  25, loss: 0.036859866231679916
epoch:  26, loss: 0

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7711313962936401
Test metrics:  R2 = 0.7358680963516235


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="scaled",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.24604372680187225
epoch:  1, loss: 0.16210508346557617
epoch:  2, loss: 0.08552583307027817
epoch:  3, loss: 0.03863321989774704
epoch:  4, loss: 0.03844670578837395
epoch:  5, loss: 0.037979234009981155
epoch:  6, loss: 0.03773849830031395
epoch:  7, loss: 0.03719930723309517
epoch:  8, loss: 0.03714906424283981
epoch:  9, loss: 0.03710031881928444
epoch:  10, loss: 0.037050679326057434
epoch:  11, loss: 0.037001874297857285
epoch:  12, loss: 0.03695238381624222
epoch:  13, loss: 0.03690285608172417
epoch:  14, loss: 0.036852359771728516
epoch:  15, loss: 0.03680231049656868
epoch:  16, loss: 0.036750588566064835
epoch:  17, loss: 0.036698468029499054
epoch:  18, loss: 0.0366445928812027
epoch:  19, loss: 0.036589816212654114
epoch:  20, loss: 0.03653297573328018
epoch:  21, loss: 0.036474764347076416
epoch:  22, loss: 0.03641476854681969
epoch:  23, loss: 0.03635333478450775
epoch:  24, loss: 0.03629011660814285
epoch:  25, loss: 0.03622497618198395
epoch:  26, los

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.690243661403656
Test metrics:  R2 = 0.6517887711524963


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="BB1",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.0949639305472374
epoch:  1, loss: 0.06886737793684006
epoch:  2, loss: 0.05465664342045784
epoch:  3, loss: 0.04686986282467842
epoch:  4, loss: 0.04259282350540161
epoch:  5, loss: 0.040239136666059494
epoch:  6, loss: 0.038941267877817154
epoch:  7, loss: 0.03822293132543564
epoch:  8, loss: 0.037824295461177826
epoch:  9, loss: 0.03760233893990517
epoch:  10, loss: 0.03747722506523132
epoch:  11, loss: 0.03740547224879265
epoch:  12, loss: 0.03736340254545212
epoch:  13, loss: 0.03733773157000542
epoch:  14, loss: 0.03732606768608093
epoch:  15, loss: 0.03729622811079025
epoch:  16, loss: 0.03727821260690689
epoch:  17, loss: 0.037272002547979355
epoch:  18, loss: 0.03724902123212814
epoch:  19, loss: 0.037234943360090256
epoch:  20, loss: 0.037223875522613525
epoch:  21, loss: 0.03720592334866524
epoch:  22, loss: 0.037201445549726486
epoch:  23, loss: 0.037178173661231995
epoch:  24, loss: 0.03716379404067993
epoch:  25, loss: 0.03714997321367264
epoch:  26, los

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7516924738883972
Test metrics:  R2 = 0.6829233765602112


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="BB2",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5247527956962585
epoch:  1, loss: 0.32701805233955383
epoch:  2, loss: 0.21650315821170807
epoch:  3, loss: 0.14873342216014862
epoch:  4, loss: 0.1064222976565361
epoch:  5, loss: 0.07995966821908951
epoch:  6, loss: 0.0635899230837822
epoch:  7, loss: 0.05351242050528526
epoch:  8, loss: 0.047304417937994
epoch:  9, loss: 0.0434773825109005
epoch:  10, loss: 0.041117649525403976
epoch:  11, loss: 0.03966224938631058
epoch:  12, loss: 0.03876422345638275
epoch:  13, loss: 0.038209494203329086
epoch:  14, loss: 0.03786627575755119
epoch:  15, loss: 0.03765315189957619
epoch:  16, loss: 0.03752027079463005
epoch:  17, loss: 0.03743645176291466
epoch:  18, loss: 0.03741257265210152
epoch:  19, loss: 0.037335120141506195
epoch:  20, loss: 0.037331171333789825
epoch:  21, loss: 0.03726242110133171
epoch:  22, loss: 0.0372195728123188
epoch:  23, loss: 0.03721168264746666
epoch:  24, loss: 0.037169698625802994
epoch:  25, loss: 0.03714335709810257
epoch:  26, loss: 0.0371

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8069145083427429
Test metrics:  R2 = 0.7715768218040466


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="quadratic",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.11612110584974289
epoch:  1, loss: 0.07773714512586594
epoch:  2, loss: 0.047370534390211105
epoch:  3, loss: 0.03799808397889137
epoch:  4, loss: 0.037539537996053696
epoch:  5, loss: 0.037494268268346786
epoch:  6, loss: 0.037411168217659
epoch:  7, loss: 0.03734216466546059
epoch:  8, loss: 0.03730323910713196
epoch:  9, loss: 0.03727000579237938
epoch:  10, loss: 0.03724297508597374
epoch:  11, loss: 0.03721834346652031
epoch:  12, loss: 0.03719504922628403
epoch:  13, loss: 0.037172503769397736
epoch:  14, loss: 0.037152670323848724
epoch:  15, loss: 0.03714103624224663
epoch:  16, loss: 0.03712468221783638
epoch:  17, loss: 0.037109870463609695
epoch:  18, loss: 0.03710159286856651
epoch:  19, loss: 0.03708517923951149
epoch:  20, loss: 0.037066005170345306
epoch:  21, loss: 0.037047579884529114
epoch:  22, loss: 0.037029486149549484
epoch:  23, loss: 0.03701179474592209
epoch:  24, loss: 0.03699532896280289
epoch:  25, loss: 0.03698454424738884
epoch:  26, los

In [14]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6854681968688965
Test metrics:  R2 = 0.6380804777145386


In [15]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="lipschitz",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3676937520503998
epoch:  1, loss: 0.23877978324890137
epoch:  2, loss: 0.1593388020992279
epoch:  3, loss: 0.11057890951633453
epoch:  4, loss: 0.08093065768480301
epoch:  5, loss: 0.06308138370513916
epoch:  6, loss: 0.05243632197380066
epoch:  7, loss: 0.046142034232616425
epoch:  8, loss: 0.042447127401828766
epoch:  9, loss: 0.04029107466340065
epoch:  10, loss: 0.03903910890221596
epoch:  11, loss: 0.038314931094646454
epoch:  12, loss: 0.037897199392318726
epoch:  13, loss: 0.037656500935554504
epoch:  14, loss: 0.03751792013645172
epoch:  15, loss: 0.03743810951709747
epoch:  16, loss: 0.03739200532436371
epoch:  17, loss: 0.03736516833305359
epoch:  18, loss: 0.03734924644231796
epoch:  19, loss: 0.037339549511671066
epoch:  20, loss: 0.03733837977051735
epoch:  21, loss: 0.03732693940401077
epoch:  22, loss: 0.037319865077733994
epoch:  23, loss: 0.03731532767415047
epoch:  24, loss: 0.037306975573301315
epoch:  25, loss: 0.037302713841199875
epoch:  26, los

In [16]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7254116535186768
Test metrics:  R2 = 0.668543815612793


In [17]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="keep",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.4395129680633545
epoch:  1, loss: 0.2705407440662384
epoch:  2, loss: 0.17085571587085724
epoch:  3, loss: 0.11251445859670639
epoch:  4, loss: 0.07892696559429169
epoch:  5, loss: 0.059947993606328964
epoch:  6, loss: 0.049415744841098785
epoch:  7, loss: 0.04366423189640045
epoch:  8, loss: 0.040565066039562225
epoch:  9, loss: 0.038913171738386154
epoch:  10, loss: 0.03804004192352295
epoch:  11, loss: 0.03758126497268677
epoch:  12, loss: 0.037341147661209106
epoch:  13, loss: 0.037215691059827805
epoch:  14, loss: 0.03715009614825249
epoch:  15, loss: 0.03711564093828201
epoch:  16, loss: 0.03709733858704567
epoch:  17, loss: 0.037087392061948776
epoch:  18, loss: 0.037081778049468994
epoch:  19, loss: 0.037078410387039185
epoch:  20, loss: 0.0370761901140213
epoch:  21, loss: 0.037074558436870575
epoch:  22, loss: 0.03707323968410492
epoch:  23, loss: 0.0370720773935318
epoch:  24, loss: 0.03707097843289375
epoch:  25, loss: 0.03706992045044899
epoch:  26, loss

In [18]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.2543618083000183
Test metrics:  R2 = 0.10730552673339844
